In [1]:
!pip install git+https://github.com/facebookresearch/ImageBind.git#egg=imagebind
!pip install setuptools<82
!pip install torchcodec

  Cloning https://github.com/facebookresearch/ImageBind.git to /tmp/pip-install-wxy_1rj3/imagebind_9fa7d55c16824f6b90fa375f2786b0da
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/ImageBind.git /tmp/pip-install-wxy_1rj3/imagebind_9fa7d55c16824f6b90fa375f2786b0da
  Resolved https://github.com/facebookresearch/ImageBind.git to commit 53680b02d7e37b19b124fa37bae4b6c98c38f5be
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/pytorchvideo.git (to revision 6cdc929315aab1b5674b6dcf73b16ec99147735f) to /tmp/pip-install-wxy_1rj3/pytorchvideo_5ae6bb86aed14343b9091649cc5f5eb8
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorchvideo.git /tmp/pip-install-wxy_1rj3/pytorchvideo_5ae6bb86aed14343b9091649cc5f5eb8
  Running command git rev-parse -q --verify 'sha^6cdc929315aab1b5674b6dcf73b16ec99147735f'
  Running command git fetch -q https://github.com/facebookresearch/pytorchvid

In [2]:
import torch
from imagebind.models.imagebind_model import ImageBindModel
from imagebind.models import imagebind_model
from imagebind import data
from imagebind.models.imagebind_model import ModalityType
import polars as pl
import random

/usr/local/lib/python3.13/dist-packages/imagebind/data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
manifest_path = "drive/MyDrive/pokemon_data/manifest.jsonl"
metadata_path = "drive/MyDrive/pokemon_data/metadata.jsonl"
device = "cuda:0" # "cpu", "cuda:0"

model = ImageBindModel(
    vision_embed_dim=1280,
    vision_num_blocks=32,
    vision_num_heads=16,
    text_embed_dim=1024,
    text_num_blocks=24,
    text_num_heads=16,
    out_embed_dim=1024,
    audio_drop_path=0.1,
    imu_drop_path=0.7,
)

In [4]:
torch.cuda.get_device_name(0)

'NVIDIA A100-SXM4-40GB'

In [5]:
model.load_state_dict(torch.load("drive/MyDrive/imagebind/.checkpoints/imagebind_huge.pth", weights_only=True))
model.eval()
model.to(device)

ImageBindModel(
  (modality_preprocessors): ModuleDict(
    (vision): RGBDTPreprocessor(
      (cls_token): tensor((1, 1, 1280), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Sequential(
          (0): PadIm2Video()
          (1): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
        )
      )
      (pos_embedding_helper): SpatioTemporalPosEmbeddingHelper(
        (pos_embed): tensor((1, 257, 1280), requires_grad=True)

      )
    )
    (text): TextPreprocessor(
      (pos_embed): tensor((1, 77, 1024), requires_grad=True)
      (mask): tensor((77, 77), requires_grad=False)

      (token_embedding): Embedding(49408, 1024)
    )
    (audio): AudioPreprocessor(
      (cls_token): tensor((1, 1, 768), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10), bias=False)
        (norm_layer): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (pos_e

In [ ]:
df = pl.read_ndjson(manifest_path)
metadata_df = pl.read_ndjson(metadata_path)

full_df = df.join(metadata_df.select(pl.exclude("name")), on="dex_num", how="left")

# use_image_flag = [True if i % 2 == 0 else False for i in range(df.shape[0])]
# use_image_flag = random.sample(use_image_flag, len(use_image_flag))
# df = df.with_columns(
#     use_image=pl.Series(use_image_flag)
# )

full_df

dex_num,name,page_title,image_path,text
str,str,str,str,str
"""0001""","""Bulbasaur""","""Bulbasaur (Pokémon)""","""data/pokemon_data/images/0001_…","""Bulbasaur is a small, quadrupe…"
"""0002""","""Ivysaur""","""Ivysaur (Pokémon)""","""data/pokemon_data/images/0002_…","""Ivysaur is a quadrupedal amphi…"
"""0003""","""Venusaur""","""Venusaur (Pokémon)""","""data/pokemon_data/images/0003_…","""Venusaur is a squat, quadruped…"
"""0004""","""Charmander""","""Charmander (Pokémon)""","""data/pokemon_data/images/0004_…","""Charmander is a bipedal, repti…"
"""0005""","""Charmeleon""","""Charmeleon (Pokémon)""","""data/pokemon_data/images/0005_…","""Charmeleon is a bipedal, repti…"
…,…,…,…,…
"""1020""","""Gouging Fire""","""Gouging Fire (Pokémon)""","""data/pokemon_data/images/1020_…","""Gouging Fire is a massive lion…"
"""1021""","""Raging Bolt""","""Raging Bolt (Pokémon)""","""data/pokemon_data/images/1021_…","""Raging Bolt is a massive quadr…"
"""1022""","""Iron Boulder""","""Iron Boulder (Pokémon)""","""data/pokemon_data/images/1022_…","""Iron Boulder is a spherical, b…"


In [ ]:
# image_df = df.filter(pl.col("use_image") == True)
# text_df = df.filter(pl.col("use_image") == False)

images = ["drive/MyDrive/" + path[5:] for path in full_df["image_path"]]
text = full_df["text"]

images[0]

'drive/MyDrive/pokemon_data/images/0001_Bulbasaur.png'

In [8]:
batch_size = 100

round = 0
all_embeddings = []
for i in range(0, len(images), batch_size):
    print("Round", round)
    cur_input = {
        ModalityType.VISION: data.load_and_transform_vision_data(images[i:i+batch_size], device),
        ModalityType.TEXT: data.load_and_transform_text(text[i:i+batch_size], device),
    }
    print("input loaded")

    with torch.amp.autocast(device, dtype=torch.float16):
        with torch.no_grad():
            embeddings = model(cur_input)

    print(embeddings)
    print()
    all_embeddings.append(embeddings)
    round += 1

Round 0
input loaded
{'vision': tensor([[ 0.0096,  0.0426,  0.0304,  ...,  0.0097, -0.0120,  0.0124],
        [ 0.0096,  0.0439,  0.0311,  ...,  0.0015, -0.0042,  0.0233],
        [ 0.0191,  0.0308,  0.0378,  ...,  0.0053, -0.0263,  0.0048],
        ...,
        [ 0.0304,  0.0396,  0.0382,  ..., -0.0067, -0.0013, -0.0002],
        [ 0.0357,  0.0291,  0.0392,  ..., -0.0172,  0.0083,  0.0047],
        [ 0.0270,  0.0211,  0.0233,  ..., -0.0536,  0.0057,  0.0158]],
       device='cuda:0'), 'text': tensor([[ 0.4884,  2.6048,  3.8958,  ...,  0.0608, -1.4891, -0.1590],
        [ 3.1568,  2.0397,  3.4674,  ...,  0.5229,  1.2787,  2.2170],
        [ 3.3461,  3.0029,  1.9368,  ..., -2.0432,  1.0891, -1.3545],
        ...,
        [ 4.7177,  3.4370,  5.1695,  ..., -0.1747,  0.3091,  3.8920],
        [ 3.6575,  5.5972,  2.1359,  ..., -1.4902,  1.9992, -0.1172],
        [-0.3060,  0.5418,  2.7420,  ..., -2.7375,  3.4397,  0.3465]],
       device='cuda:0')}

Round 1
input loaded
{'vision': tensor([[

# Ingesting

In [10]:
!pip install lancedb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.8/406.8 kB 41.1 MB/s eta 0:00:00


In [11]:
all_embeddings[0]['vision'].shape

torch.Size([100, 1024])

In [12]:
all_embeddings[0]['vision'][0:2].shape

torch.Size([2, 1024])

In [ ]:
#image_df

dex_num,name,page_title,image_path,text,use_image
str,str,str,str,str,bool
"""0004""","""Charmander""","""Charmander (Pokémon)""","""data/pokemon_data/images/0004_…","""Charmander is a bipedal, repti…",true
"""0006""","""Charizard""","""Charizard (Pokémon)""","""data/pokemon_data/images/0006_…","""Charizard is a draconic , bipe…",true
"""0012""","""Butterfree""","""Butterfree (Pokémon)""","""data/pokemon_data/images/0012_…","""Butterfree is a lepidopteran i…",true
"""0013""","""Weedle""","""Weedle (Pokémon)""","""data/pokemon_data/images/0013_…","""Weedle is a larval insect Poké…",true
"""0014""","""Kakuna""","""Kakuna (Pokémon)""","""data/pokemon_data/images/0014_…","""Kakuna is a yellow, insect coc…",true
…,…,…,…,…,…
"""1010""","""Iron Leaves""","""Iron Leaves (Pokémon)""","""data/pokemon_data/images/1010_…","""Iron Leaves is a mechanical Po…",true
"""1012""","""Poltchageist""","""Poltchageist (Pokémon)""","""data/pokemon_data/images/1012_…","""Poltchageist is a Pokémon made…",true
"""1013""","""Sinistcha""","""Sinistcha (Pokémon)""","""data/pokemon_data/images/1013_…","""Sinistcha is a Pokémon made of…",true


In [ ]:
#text_df

dex_num,name,page_title,image_path,text,use_image
str,str,str,str,str,bool
"""0001""","""Bulbasaur""","""Bulbasaur (Pokémon)""","""data/pokemon_data/images/0001_…","""Bulbasaur is a small, quadrupe…",false
"""0002""","""Ivysaur""","""Ivysaur (Pokémon)""","""data/pokemon_data/images/0002_…","""Ivysaur is a quadrupedal amphi…",false
"""0003""","""Venusaur""","""Venusaur (Pokémon)""","""data/pokemon_data/images/0003_…","""Venusaur is a squat, quadruped…",false
"""0005""","""Charmeleon""","""Charmeleon (Pokémon)""","""data/pokemon_data/images/0005_…","""Charmeleon is a bipedal, repti…",false
"""0007""","""Squirtle""","""Squirtle (Pokémon)""","""data/pokemon_data/images/0007_…","""Squirtle is a small reptilian …",false
…,…,…,…,…,…
"""1019""","""Hydrapple""","""Hydrapple (Pokémon)""","""data/pokemon_data/images/1019_…","""Hydrapple is a Pokémon compose…",false
"""1020""","""Gouging Fire""","""Gouging Fire (Pokémon)""","""data/pokemon_data/images/1020_…","""Gouging Fire is a massive lion…",false
"""1022""","""Iron Boulder""","""Iron Boulder (Pokémon)""","""data/pokemon_data/images/1022_…","""Iron Boulder is a spherical, b…",false


In [ ]:
to_ingest = []

round = 0
for i in range(0, len(images), batch_size):
    print("Round", round)

    # vision and text set set
    data_part = full_df.slice(i, i+batch_size).to_dict(as_series=False)
    vals = list(data_part.values()) + [all_embeddings[round]['vision'].tolist(), all_embeddings[round]['text'].tolist()]
    keys = list(data_part.keys()) + ["image_vector", "text_vector"]

    data_ingest = [dict(zip(keys, values)) for values in zip(*vals)]

    to_ingest += data_ingest
    round += 1

print(len(to_ingest))
to_ingest[0]

Round 0
Round 1
Round 2
Round 3
Round 4
Round 5
Round 6
Round 7
Round 8
Round 9
1000


{'dex_num': '0001',
 'name': 'Bulbasaur',
 'page_title': 'Bulbasaur (Pokémon)',
 'image_path': 'data/pokemon_data/images/0001_Bulbasaur.png',
 'text': "Bulbasaur is a small, quadrupedal amphibian Pokémon that has blue-green skin with darker patches. It has red eyes with white pupils, pointed, ear-like structures on top of its head, and a short, blunt snout with a wide mouth. Small, pointed teeth are visible in the upper jaw when the mouth is open. Each of its thick legs ends with three sharp claws. On Bulbasaur's back is a green plant bulb that conceals two slender, tentacle-like vines, which grow from a seed planted there at birth. The bulb also provides it with energy through photosynthesis and from the nutrient-rich seeds contained within.",
 'image_vector': [0.009552144445478916,
  0.0426461435854435,
  0.03044852800667286,
  -0.015861453488469124,
  0.025874421000480652,
  0.019172558560967445,
  -0.036615606397390366,
  -0.006718929391354322,
  -0.009552144445478916,
  0.05602711

In [1]:
import lancedb

uri = "pokemon_db"
db = lancedb.connect(uri)

In [15]:
table = db.create_table("pokemon", data=to_ingest, mode="overwrite")

In [16]:
len(all_embeddings[0]['text'][0].tolist())

1024

In [20]:
query_vector = all_embeddings[0]['text'][0].tolist()
#query_vector = all_embeddings[2]['text'][95].tolist()

result = (
    table.search(query_vector, vector_column_name="text_vector")
    .select(["name", "_distance"])
    .limit(10)
    .to_polars()
)
print(result)

shape: (10, 2)
┌────────────┬─────────────┐
│ name       ┆ _distance   │
│ ---        ┆ ---         │
│ str        ┆ f32         │
╞════════════╪═════════════╡
│ Bulbasaur  ┆ 0.0         │
│ Venusaur   ┆ 4113.521973 │
│ Ivysaur    ┆ 4364.966309 │
│ Charmander ┆ 5354.098145 │
│ Squirtle   ┆ 5372.14209  │
│ Leafeon    ┆ 5392.914062 │
│ Glaceon    ┆ 6174.446289 │
│ Snorlax    ┆ 6400.994629 │
│ Charizard  ┆ 6464.520996 │
│ Shroomish  ┆ 6626.084961 │
└────────────┴─────────────┘


In [21]:
!zip -r pokemon_db.zip pokemon_db

  adding: pokemon_db/ (stored 0%)
  adding: pokemon_db/pokemon.lance/ (stored 0%)
  adding: pokemon_db/pokemon.lance/data/ (stored 0%)
  adding: pokemon_db/pokemon.lance/data/101101100110010111000011189889473ba2f016ff4865e45f.lance (deflated 15%)
  adding: pokemon_db/pokemon.lance/_transactions/ (stored 0%)
  adding: pokemon_db/pokemon.lance/_transactions/0-7a153862-bc47-4926-b44d-c20cfd16481a.txn (deflated 43%)
  adding: pokemon_db/pokemon.lance/_versions/ (stored 0%)
  adding: pokemon_db/pokemon.lance/_versions/latest_version_hint.json (stored 0%)
  adding: pokemon_db/pokemon.lance/_versions/18446744073709551614.manifest (deflated 63%)


In [2]:
db.list_tables()

ListTablesResponse(context=None, tables=[], page_token=None)

In [ ]:
#db.drop_table("pokemon")